# Imports

In [66]:
from datetime import datetime as DT
from datetime import timedelta
from pathlib import Path
import os
from WS_Mdl.core import Mdl_N, Sep, sprint
from WS_Mdl.imod.mf6.bin import to_DF
import pandas as pd
import WS_Mdl.core.df # noqa: F401
from WS_Mdl.core.defaults import CRS
import geopandas as gpd
from shapely.geometry import Point
from WS_Mdl.imod.mf6.write import add_OBS_to_MF_In

# Args

In [39]:
Pa_Shp =  r'G:\models\NBr\PoP\common\Pgn\Chaamse_beek\catchment_chaamsebeek_ulvenhout.shp' # Shapefile to clip elements inside.
MdlN = 'NBr54'
MdlN_B = True
Pkg = 'DRN'
Opt = """BEGIN OPTIONS
	DIGITS 4
	PRINT_INPUT
END OPTIONS\n
"""

# Init

In [ ]:
sprint(Sep, verbose_out=False)
M = Mdl_N(MdlN)

if MdlN_B is True:
    MdlN_B = M.B
M_B = Mdl_N(MdlN_B)

Xmin, Ymin, Xmax, Ymax = [float(i) for i in M.INI['WINDOW'].split(',')]
cellsize = float(M.INI.CELLSIZE)

----------------------------------------------------------------------------------------------------



# Load Shp

In [41]:
if Pa_Shp is not None:
    GDF_Shp = gpd.read_file(Pa_Shp)
    print(f"Loaded shapefile with {len(GDF_Shp)} features")
    print(f"CRS: {GDF_Shp.crs}")
    print(f"Bounds: {GDF_Shp.bounds}")
    GDF_Shp.crs = CRS

Loaded shapefile with 1 features
CRS: None
Bounds:           minx         miny         maxx         maxy
0  113434.7999  387938.1329  124760.5671  395868.0663


# Load DF

In [63]:
d = {f.parent.stem: {'path': f,
                 'DF': to_DF(f, Pkg)} for f in M.Pa.Sim_In.rglob(f'{Pkg.lower()}*.bin')}

In [68]:
d

{'drn-1': {'path': WindowsPath('G:/models/NBr/Sim/NBr54/modflow6/imported_model/drn-1/drn-0.bin'),
  'DF':        k    i    j       elev       cond      N         X         Y  \
  0      1    1    1   3.214980  18.939394      1  113112.5  396187.5   
  1      1    1    2   3.253483  18.939394      2  113137.5  396187.5   
  2      1    1    3   3.202580  18.939394      3  113162.5  396187.5   
  3      1    1    4   3.239084  18.939394      4  113187.5  396187.5   
  4      1    1    5   3.196262  18.939394      5  113212.5  396187.5   
  ...   ..  ...  ...        ...        ...    ...       ...       ...   
  22742  1  344  352  18.786858   8.802817  22743  121887.5  387612.5   
  22743  1  344  353  18.814594   8.802817  22744  121912.5  387612.5   
  22744  1  344  354  19.013451   8.802817  22745  121937.5  387612.5   
  22745  1  344  355  19.002575   8.802817  22746  121962.5  387612.5   
  22746  1  344  356  18.797590   8.802817  22747  121987.5  387612.5   
  
                

In [69]:
for S in d:
    d[S]['DF']['N'] = d[S]['DF']['i'].index + 1
    Sys = S.split('-')[-1]
    d[S]['DF'] = d[S]['DF'].ws.Calc_XY(Xmin=Xmin, Ymax=Ymax, cellsize=cellsize)
    
    # Create geometry for DRN points
    d[S]['DF']['geometry'] = d[S]['DF'].apply(lambda row: Point(row['X'], row['Y']), axis=1)
    GDF = gpd.GeoDataFrame(d[S]['DF'], crs=CRS)

    # Store init counts before filtering
    N_init = len(d[S]['DF'])

    # Clip to shapefile    
    GDF = gpd.sjoin(GDF, GDF_Shp, how='inner', predicate='within')

    if not GDF.empty:
        # Write
        DF_w = pd.DataFrame()
        DF_w['obsname'] = GDF.apply(lambda row: f"{Pkg}_L{int(row['k'])}_R{int(row['i'])}_C{int(row['j'])}", axis=1)
        DF_w['obstype'] = Pkg
        DF_w['id'] = GDF.apply(lambda row: f"{int(row['k'])} {int(row['i'])} {int(row['j'])}", axis=1)

        Fi = f"{MdlN}.{S}.OBS6"
        with open(M.Pa.Sim_In / Fi, 'w') as f:
            f.write(Opt)
            f.write(f"BEGIN CONTINUOUS FILEOUT {Pkg}_OBS_Sys_{Sys}.CSV\n")
            f.write(DF_w.ws.to_MF_block())
            f.write("END CONTINUOUS FILEOUT\n")

        # Add to MF_In
        add_OBS_to_MF_In(
            str_OBS = f' OBS6 FILEIN ./imported_model/{Fi}', Pa=M.Pa.Sim_In / f'{S}.{Pkg.lower()}', iMOD5=False
        )

🟢 - Added OBS to G:\models\NBr\Sim\NBr54\modflow6\imported_model\drn-1.drn
🟢 - Added OBS to G:\models\NBr\Sim\NBr54\modflow6\imported_model\drn-2.drn
